# 이오스시스템 메인 페이지 스크래핑 데이터 — 기초 EDA 리포트

- 데이터: `eos_main_scrape.csv` (`스크랩핑실습.ipynb`로 수집)
- 수집 대상: https://www.이오스시스템.com/default/ 메인 페이지
- 수집 방법: HTML 텍스트(BeautifulSoup) + 이미지 속 글자(EasyOCR) + 링크된 카탈로그 PDF 텍스트(PyMuPDF)

In [ ]:
import re
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'Malgun Gothic'   # 한글 폰트 (Windows)
plt.rcParams['axes.unicode_minus'] = False       # 마이너스 기호 깨짐 방지

df = pd.read_csv('eos_main_scrape.csv', encoding='utf-8-sig')
df.head()

## 1. 데이터 개요

In [ ]:
print('행/열:', df.shape)
print('중복행:', df.duplicated().sum())
print()
print('수집 종류(type)별 행 수')
print(df['type'].value_counts())

- 행/열: (247행, 5열) → 완전히 같은 중복행은 0개
- 수집 종류: `html_text` 118행 / `image_ocr` 105행 / `pdf_text` 24행
- 주요 컬럼
  - `type`: 텍스트를 어디서 가져왔는지 (HTML / 이미지 OCR / PDF)
  - `source`: 출처 URL (HTML은 페이지 주소, 이미지는 이미지 주소, PDF는 PDF 주소)
  - `order`: 순서 (HTML은 줄 번호, 이미지는 이미지 번호, PDF는 쪽 번호)
  - `content`: 실제 텍스트
  - `note`: 메모 (OCR로 찾은 글자 상자 수, 건너뛴 이유 등)

## 2. 구조 진단 (shape / info / describe)

In [ ]:
df.info()

In [ ]:
# 텍스트 길이(글자 수) 파생 변수
df['length'] = df['content'].fillna('').str.len()
df.groupby('type')['length'].describe().round(1)

- 자료형 요약: `order`만 정수(int), 나머지 4개 열은 모두 문자열
- 수치 요약에서 눈에 띈 점
  - `pdf_text`는 한 행이 PDF 한 쪽이라 평균 약 730자로 매우 길다 (최대 1,984자 = 22쪽 회사 소개)
  - `html_text`는 한 행이 한 줄이라 평균 약 12자로 짧다
  - `image_ocr`은 중앙값이 **0자** → 이미지 절반 이상에서 글자를 읽지 못했다
  - 같은 '1행'이라도 종류마다 텍스트 단위(줄 / 이미지 / 쪽)가 달라서 행 수끼리 바로 비교하면 안 된다

## 3. 품질 진단 (결측 / 중복 / 표기)

In [ ]:
print('열별 결측 수')
print(df.isna().sum())
print()
print('type별 content 결측 수')
print(df.groupby('type')['content'].apply(lambda s: s.isna().sum()))

In [ ]:
# OCR 결과 메모 확인
df.loc[df['type'] == 'image_ocr', 'note'].value_counts()

In [ ]:
# 내용이 같은 텍스트(중복 내용) 확인
dup = df[df['content'].notna() & df['content'].duplicated(keep=False)]
dup.sort_values('content')[['type', 'source', 'content']]

In [ ]:
# OCR 오인식 예시: 한글/영문/숫자가 아닌 기호가 섞이거나 의미 없는 영문 조각
ocr = df[(df['type'] == 'image_ocr') & df['content'].notna()]
ocr[['source', 'content', 'note']].head(15)

- 결측: `content` 55건 → **모두 `image_ocr`** (HTML·PDF는 결측 0)
  - 47건: OCR을 돌렸지만 글자를 못 찾음 (`0 boxes`) — 아이콘·제품 사진처럼 원래 글자가 없는 이미지가 대부분
  - 8건: 이미지가 너무 작아 OCR 생략 (`skipped: too small`)
- `note` 결측 142건은 HTML·PDF 행이라 메모가 필요 없어서 비어 있는 것 → 문제 아님
- 중복: 완전 중복행 0건, 내용만 같은 행 3쌍 (`Hansot`, `사브올데이`, `연안식당`)
  - 같은 브랜드 로고가 페이지의 두 섹션(로고 목록 / 슬라이더)에 각각 있어서 생긴 것
- 표기 문제 (OCR 오인식)
  - 철자 오류: `폐리카나`(→ 페리카나), `설치시례`(→ 설치사례), `인전본사`(→ 인천본사), `국대떡 볶이`(띄어쓰기)
  - 의미 없는 조각: `Wsos 0`, `JC`, `T`, `bhz9`
  - 로고 `EOS`를 `cos`, `coscOS`, `6OS` 등으로 잘못 읽음

## 4. 패턴 (시각화에서 읽은 것)

In [ ]:
# (1) 수집 종류별 행 수 vs 전체 글자 수
summary = df.groupby('type').agg(행수=('content', 'size'), 글자수=('length', 'sum'))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
summary['행수'].plot.bar(ax=axes[0], color='steelblue', rot=0, title='종류별 행 수')
summary['글자수'].plot.bar(ax=axes[1], color='darkorange', rot=0, title='종류별 전체 글자 수')
for ax in axes:
    for p in ax.patches:
        ax.annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width() / 2, p.get_height()),
                    ha='center', va='bottom')
plt.tight_layout()
plt.show()

print((summary['글자수'] / summary['글자수'].sum() * 100).round(1).astype(str) + '%')

In [ ]:
# (2) 이미지가 있는 섹션별 OCR 성공률
img = df[df['type'] == 'image_ocr'].copy()
img['section'] = img['source'].str.extract(r'design/([^/]+)/')[0].fillna('게시판(설치사례·리뷰)')
img['ocr_ok'] = img['content'].notna()

sec = img.groupby('section').agg(이미지수=('ocr_ok', 'size'), 성공=('ocr_ok', 'sum'))
sec['성공률(%)'] = (sec['성공'] / sec['이미지수'] * 100).round(0)
sec = sec.sort_values('이미지수', ascending=False)
display(sec)

ax = sec[['성공', '이미지수']].plot.barh(figsize=(8, 4), color=['seagreen', 'lightgray'])
ax.set_title('섹션별 이미지 수와 OCR 성공 수')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# (3) PDF 쪽별 글자 수
pdf = df[df['type'] == 'pdf_text']
ax = pdf.plot.bar(x='order', y='length', figsize=(11, 3.5), legend=False, color='slateblue', rot=0)
ax.axhline(pdf['length'].mean(), color='red', linestyle='--', label=f"평균 {pdf['length'].mean():.0f}자")
ax.set_xlabel('PDF 쪽 번호')
ax.set_ylabel('글자 수')
ax.set_title('카탈로그 PDF 쪽별 글자 수')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# (4) 전체 텍스트에서 많이 나온 한글 단어 TOP 20 (2글자 이상)
words = re.findall(r'[가-힣]{2,}', ' '.join(df['content'].dropna()))
top = pd.Series(Counter(words)).sort_values(ascending=False).head(20)

ax = top.sort_values().plot.barh(figsize=(8, 6), color='teal')
ax.set_title('한글 단어 빈도 TOP 20')
ax.set_xlabel('등장 횟수')
plt.tight_layout()
plt.show()

- 분량: 행 수는 HTML(118)이 가장 많지만, **글자 수는 PDF가 약 88%**를 차지한다
  → 텍스트 정보 대부분은 웹 페이지가 아니라 카탈로그 PDF에 있다
- OCR 성공률: 섹션에 따라 크게 다르다
  - 게시판(설치사례·리뷰) 이미지 15개 → **100%** 성공, 사진 위에 큰 글씨가 올라간 이미지
  - `banner`·`slider03`(거래처 브랜드 로고) → 약 67~68% 성공, 디자인 글씨라 일부는 놓치거나 틀림
  - `slider04`(제품 이미지) → 12%, `cnp`(더보기 버튼 등 아이콘 25개)·`footer_03`(로고·SNS 아이콘) → **0%**
    → 원래 글자가 거의 없는 이미지라 실패가 아니라 정상
- PDF: 1~3쪽(표지·목차)과 21·24쪽은 짧고, 대부분 쪽이 600~1,000자로 고르다. 22쪽(회사 소개)이 1,984자로 가장 길다
- 단어 빈도: `제품`, `초음파`, `절감`, `업소용`, `매장`, `결제`, `비용`, `식기세척기`가 상위
  → 이 회사는 **업소용 주방가전(초음파 식기세척기)과 매장 결제 시스템**을 중심으로, **비용 절감**을 핵심 메시지로 내세운다

## 5. 다음 분석 제안

- OCR 결측 55건: 아이콘·사진 이미지는 원래 글자가 없으므로 결측을 채우지 말고 **분석에서 제외**한다
  (`cnp`, `footer_03` 섹션은 다음 수집부터 OCR 대상에서 빼면 시간도 줄어든다)
- OCR 오인식: 신뢰도 기준을 0.3 → 0.5로 올려 의미 없는 조각(`Wsos 0`, `T` 등)을 줄이고,
  자주 틀리는 단어(`설치시례`→`설치사례`, `cos`/`6OS`→`EOS`)는 치환 사전으로 정리한다
- 중복 3쌍: 같은 로고가 두 번 나온 것이므로 브랜드 목록을 만들 때는 `content` 기준으로 하나만 남긴다
- 단위 통일: 줄 / 이미지 / 쪽 단위가 섞여 있으므로, 텍스트 분석을 할 때는 PDF를 줄 단위로 나눠서 HTML과 단위를 맞춘다
- 추가로 보고 싶은 것
  - OCR로 읽은 브랜드명만 모아 **거래처(가맹 브랜드) 목록** 만들기
  - PDF 쪽별 제품명과 키워드를 묶어 **제품별 핵심 문구** 비교하기
  - 웹 페이지(HTML)와 카탈로그(PDF)에서 강조하는 단어 차이 비교하기